In [7]:
## Defining packages
import os
import shutil
import pathlib
from datetime import datetime
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
import piexif
import re

## Defining Static Variables
datetime_original_tag = 36867
datetime_digitized_tag = 36868
datetime_tag = 306

In [14]:
source_base = "C:/Users/laram/Pictures/20170223 - 2019 Renamed Snapchats/"
destination_base = "C:/Users/laram/Pictures/20170223 - 2019 Renamed Snapchats with EXIF/"

In [15]:
def filename_timestamp_splitter(filename):
    
    # extract only digits
    digits = re.sub(r"\D", "", filename)

    # must be exactly 14 digits
    if len(digits) < 14:
        return False

    digits = digits[:14]
    return digits

In [16]:
def format_exif_timestamp(datetime_string):
    
    parsed = datetime.strptime(datetime_string, "%Y%m%d%H%M%S")
    exif_format = parsed.strftime("%Y:%m:%d %H:%M:%S")
    
    return exif_format

In [17]:
def contains_real_datetime(filename):
    
    # extract only digits
    digits = re.sub(r"\D", "", filename)

    # must be exactly 14 digits
    if len(digits) < 14:
        return False

    digits = digits[:14]

    try:
        datetime.strptime(digits, "%Y%m%d%H%M%S")
        return True
    except ValueError:
        return False

In [18]:
def has_exif_datetime(image_path):
    
    img = Image.open(image_path)
    exif_data = img._getexif()
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    img_datetime = exif_data.get(datetime_tag)

    true_datetime = original_datetime or digitized_datetime or img_datetime
    
    if true_datetime:
        return True
    else:
        return False

# Steps

1. check if filename has valid datetime name (if not = ignore, no way to set exif data)
2. check if there is exif data for main datetime original (if there is = ignore, no reason to add datetime data)
   - but check first that it matches the filename, flag it if it doesnt
3. check if there is exif data for other datetime values (if there is, must do some validation)
   - if there is no exif datetime data at all -> easy = use filename datetime
   - if there is a datetime value -> check that it matches the filename
      - if it does match the filename -> good to set all datetimes to that value
      - if it does not match the filename -> ignore and flag the file as contradictory
4. copy image to destination
6. add all datetime values with the one found in filename (if previous checks have passed)
7. do some validation that the datetime value has changed and maybe some validation that the rest of the exif hasn't been altered? if not overkill?
8. once the file has been moved (check it exists) and there's a datetime original value, delete the original image from the source path

Note: This code only works reliably for JPG/JPEG files.

In [19]:
contradictary_files = {}

folder = Path(source_base)
for file in folder.rglob("*"):
    
    suffix = Path(str(file)).suffix
    if suffix.upper() not in ['.JPG', '.JPEG']:
        continue
    
    # Defining file-specific variables
    file_source_path = str(file)
    filename = Path(file_source_path).name
    
    print("Reading:", filename)
    
    try:
        img = Image.open(file_source_path)
    except:
        print("Error opening image:", file_source_path)
        continue
        
    exif_data = img._getexif()
    if not exif_data:
        print("Error opening EXIF, need to create manually:", filename)
        continue
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    plain_datetime = exif_data.get(datetime_tag)
    filename_datetime = filename_timestamp_splitter(filename[:17])
    
    
    # Step 1: Ignore if there is no datetime found in the filename
    if not contains_real_datetime(filename[:17]):
        print("   Step 1: Failed - filename does not contain datetime data -", file_source_path)
        continue

    filename_datetime_formatted = format_exif_timestamp(filename_datetime)
    
    
    # Step 2: Ignore if there is already a datetime found in the EXIF data (but with consolidation checks)
    if has_exif_datetime(file_source_path):
        
        print("   Step 2: Failed - EXIF data already contains datetime data -", file_source_path)
        
        # Step 2.5: Flag the file if the filename datetime is different from the EXIF Datetime original
        if original_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = original_datetime
            print("   Step 2.5: Failed")

        continue
    
    # Step 3: Check that other exif datetime values don't differ from the filename datetime
    if digitized_datetime or plain_datetime:
        other_exif_datetime = digitized_datetime or plain_datetime
        if other_exif_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = other_exif_datetime
            print("   Step 3: Failed - Contradictory dates found -", file_source_path)
            continue

            
    # Happy that the filename will be the datetime value for the exif data
    final_exif_datetime = filename_datetime_formatted
    
    # Step 4: Copy image to the destination
    file_dest_path = destination_base + filename
    try:
        shutil.copyfile(file_source_path, file_dest_path)
    except:
        print("Error copying file: " + file_path)
        print("   Step 4: Failed (Error copying file: " + file_path + ")")
        continue


    # Step 5: Add datetime values to exif data if they don't already exist
    with Image.open(file_dest_path) as new_img:
        new_exif_data = new_img._getexif()
        new_exif_dict = piexif.load(new_img.info.get("exif", b""))

        new_exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal]= final_exif_datetime

        try:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized]
        except:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized] = final_exif_datetime

        try:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime]
        except:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime] = final_exif_datetime

        exif_bytes = piexif.dump(new_exif_dict)

        new_img.save(file_dest_path, exif=exif_bytes)
        img.close()
        new_img.close()
    
    # Step 6: Validation
    if Path(file_dest_path).exists():
        
        with Image.open(file_dest_path) as verify_img:
            new_exif_data = verify_img._getexif()

        if new_exif_data.get(datetime_original_tag):
            os.remove(file_source_path)
        else:
            print("Datetime data not found in EXIF:", file_dest_path)
            continue
    else:
        print("Copy failed, not deleting:", source)
        continue
    


Reading: 2017-02-23_211616_0_main.jpg
Reading: 2017-02-23_211626_0_main.jpg
Reading: 2017-02-24_142421_0_main.jpg
Reading: 2017-02-24_142747_0_main.jpg
Reading: 2017-02-25_145606_0_main.jpg
Reading: 2017-02-25_224054_0_main.jpg
Reading: 2017-02-25_224335_0_main.jpg
Reading: 2017-02-25_235909_0_main.jpg
Reading: 2017-02-26_003253_0_main.jpg
Reading: 2017-02-27_202356_0_main.jpg
Reading: 2017-02-28_105554_0_main.jpg
Reading: 2017-03-01_124220_0_main.jpg
Reading: 2017-03-01_140023_0_main.jpg
Reading: 2017-03-01_154237_0_main.jpg
Reading: 2017-03-02_120447_0_main.jpg
Reading: 2017-03-02_194444_0_main.jpg
Reading: 2017-03-02_194500_0_main.jpg
Reading: 2017-03-02_195818_0_main.jpg
Reading: 2017-03-02_201330_0_main.jpg
Reading: 2017-03-03_115924_0_main.jpg
Reading: 2017-03-03_210742_0_main.jpg
Reading: 2017-03-03_212657_0_main.jpg
Reading: 2017-03-03_212720_0_main.jpg
Reading: 2017-03-04_134123_0_main.jpg
Reading: 2017-03-04_193725_0_main.jpg
Reading: 2017-03-04_193738_0_main.jpg
Reading: 201

Reading: 2017-04-13_164055_0_main.jpg
Reading: 2017-04-13_164250_0_main.jpg
Reading: 2017-04-13_164343_0_main.jpg
Reading: 2017-04-14_174717_0_main.jpg
Reading: 2017-04-14_193318_0_main.jpg
Reading: 2017-04-14_220117_0_main.jpg
Reading: 2017-04-15_144655_0_main.jpg
Reading: 2017-04-15_174025_0_main.jpg
Reading: 2017-04-15_174359_0_main.jpg
Reading: 2017-04-15_180727_0_main.jpg
Reading: 2017-04-15_180806_0_main.jpg
Reading: 2017-04-15_180831_0_main.jpg
Reading: 2017-04-15_181407_0_main.jpg
Reading: 2017-04-15_182027_0_main.jpg
Reading: 2017-04-15_182510_0_main.jpg
Reading: 2017-04-15_183505_0_main.jpg
Reading: 2017-04-15_183514_0_main.jpg
Reading: 2017-04-15_183552_0_main.jpg
Reading: 2017-04-15_193321_0_main.jpg
Reading: 2017-04-16_144516_0_main.jpg
Reading: 2017-04-16_144545_0_main.jpg
Reading: 2017-04-16_214809_0_main.jpg
Reading: 2017-04-17_174717_0_main.jpg
Reading: 2017-04-17_232421_0_main.jpg
Reading: 2017-04-18_105501_0_main.jpg
Reading: 2017-04-18_230323_0_main.jpg
Reading: 201

Reading: 2017-05-17_101436_0_main.jpg
Reading: 2017-05-17_101522_0_main.jpg
Reading: 2017-05-17_165511_0_main.jpg
Reading: 2017-05-17_165521_0_main.jpg
Reading: 2017-05-17_185948_0_main.jpg
Reading: 2017-05-17_191829_0_main.jpg
Reading: 2017-05-17_212128_0_main.jpg
Reading: 2017-05-17_220059_0_main.jpg
Reading: 2017-05-17_220733_0_main.jpg
Reading: 2017-05-18_080409_0_main.jpg
Reading: 2017-05-18_081024_0_main.jpg
Reading: 2017-05-18_140017_0_main.jpg
Reading: 2017-05-18_152301_0_main.jpg
Reading: 2017-05-18_164653_0_main.jpg
Reading: 2017-05-18_172811_0_main.jpg
Reading: 2017-05-18_173044_0_main.jpg
Reading: 2017-05-18_173052_0_main.jpg
Reading: 2017-05-18_174026_0_main.jpg
Reading: 2017-05-19_135549_0_main.jpg
Reading: 2017-05-19_135708_0_main.jpg
Reading: 2017-05-19_155301_0_main.jpg
Reading: 2017-05-19_164128_0_main.jpg
Reading: 2017-05-19_164726_0_main.jpg
Reading: 2017-05-19_164732_0_main.jpg
Reading: 2017-05-19_165409_0_main.jpg
Reading: 2017-05-19_175744_0_main.jpg
Reading: 201

Reading: 2017-06-08_201948_0_main.jpg
Reading: 2017-06-08_210922_0_main.jpg
Reading: 2017-06-09_135548_0_main.jpg
Reading: 2017-06-09_182658_0_main.jpg
Reading: 2017-06-09_184335_0_main.jpg
Reading: 2017-06-10_130553_0_main.jpg
Reading: 2017-06-10_133925_0_main.jpg
Reading: 2017-06-10_133945_0_main.jpg
Reading: 2017-06-10_134134_0_main.jpg
Reading: 2017-06-10_142204_0_main.jpg
Reading: 2017-06-10_142728_0_main.jpg
Reading: 2017-06-10_154053_0_main.jpg
Reading: 2017-06-10_154107_0_main.jpg
Reading: 2017-06-10_154119_0_main.jpg
Reading: 2017-06-11_121616_0_main.jpg
Reading: 2017-06-11_121624_0_main.jpg
Reading: 2017-06-11_121631_0_main.jpg
Reading: 2017-06-11_121829_0_main.jpg
Reading: 2017-06-11_143342_0_main.jpg
Reading: 2017-06-11_161446_0_main.jpg
Reading: 2017-06-11_172035_0_main.jpg
Reading: 2017-06-11_192348_0_main.jpg
Reading: 2017-06-11_194656_0_main.jpg
Reading: 2017-06-11_194722_0_main.jpg
Reading: 2017-06-11_194832_0_main.jpg
Reading: 2017-06-11_194851_0_main.jpg
Reading: 201

Reading: 2017-07-02_181330_0_main.jpg
Reading: 2017-07-02_181342_0_main.jpg
Reading: 2017-07-02_181542_0_main.jpg
Reading: 2017-07-02_181549_0_main.jpg
Reading: 2017-07-02_181602_0_main.jpg
Reading: 2017-07-02_192056_0_main.jpg
Reading: 2017-07-02_222353_0_main.jpg
Reading: 2017-07-02_223540_0_main.jpg
Reading: 2017-07-03_021713_0_main.jpg
Reading: 2017-07-03_110736_0_main.jpg
Reading: 2017-07-04_115127_0_main.jpg
Reading: 2017-07-04_115258_0_main.jpg
Reading: 2017-07-04_120247_0_main.jpg
Reading: 2017-07-04_124608_0_main.jpg
Reading: 2017-07-04_124637_0_main.jpg
Reading: 2017-07-04_194102_0_main.jpg
Reading: 2017-07-05_165756_0_main.jpg
Reading: 2017-07-05_212821_0_main.jpg
Reading: 2017-07-05_212833_0_main.jpg
Reading: 2017-07-06_103358_0_main.jpg
Reading: 2017-07-06_104724_0_main.jpg
Reading: 2017-07-06_110331_0_main.jpg
Reading: 2017-07-06_195534_0_main.jpg
Reading: 2017-07-07_235953_0_main.jpg
Reading: 2017-07-08_182702_0_main.jpg
Reading: 2017-07-08_182729_0_main.jpg
Reading: 201

...

Reading: 2021-06-03_134117_0_main.jpg
Reading: 2021-06-03_194929_0_main.jpg
Reading: 2021-06-03_194947_0_main.jpg
Reading: 2021-06-04_092438_0_main.jpg
Reading: 2021-06-04_110450_0_main.jpg
Reading: 2021-06-04_112418_0_main.jpg
Reading: 2021-06-04_125540_0_main.jpg
Reading: 2021-06-04_125554_0_main.jpg
Reading: 2021-06-04_125718_0_main.jpg
Reading: 2021-06-04_125944_0_main.jpg
Reading: 2021-06-05_111129_0_main.jpg
Reading: 2021-06-05_111151_0_main.jpg
Reading: 2021-06-05_170039_0_main.jpg
Reading: 2021-06-05_191804_0_main.jpg
Reading: 2021-06-06_035531_0_main.jpg
Reading: 2021-06-06_044347_0_main.jpg
Reading: 2021-06-06_091857_0_main.jpg
Reading: 2021-06-06_091904_0_main.jpg
Reading: 2021-06-06_091910_0_main.jpg
Reading: 2021-06-06_180028_0_main.jpg
Reading: 2021-06-07_174003_0_main.jpg
Reading: 2021-06-07_183333_0_main.jpg
Reading: 2021-06-07_185650_0_main.jpg
Reading: 2021-06-08_081158_0_main.jpg
Reading: 2021-06-08_081332_0_main.jpg
Reading: 2021-06-09_060617_0_main.jpg
Reading: 202

Reading: 2021-07-27_200819_0_main.jpg
Reading: 2021-07-27_231838_0_main.jpg
Reading: 2021-07-28_160530_0_main.jpg
Reading: 2021-07-30_165106_0_main.jpg
Reading: 2021-07-30_165138_0_main.jpg
Reading: 2021-07-30_224434_0_main.jpg
Reading: 2021-07-31_191931_0_main.jpg
Reading: 2021-07-31_212117_0_main.jpg
Reading: 2021-08-01_021541_0_main.jpg
Reading: 2021-08-01_021552_0_main.jpg
Reading: 2021-08-02_191237_0_main.jpg
Reading: 2021-08-02_221509_0_main.jpg
Reading: 2021-08-03_131549_0_main.jpg
Reading: 2021-08-03_160121_0_main.jpg
Reading: 2021-08-04_132702_0_main.jpg
Reading: 2021-08-05_142014_0_main.jpg
Reading: 2021-08-07_200322_0_main.jpg
Reading: 2021-08-08_231331_0_main.jpg
Reading: 2021-08-09_120126_0_main.jpg
Reading: 2021-08-09_164123_0_main.jpg
Reading: 2021-08-09_164128_0_main.jpg
Reading: 2021-08-10_140434_0_main.jpg
Reading: 2021-08-10_140649_0_main.jpg
Reading: 2021-08-10_163108_0_main.jpg
Reading: 2021-08-11_083433_0_main.jpg
Reading: 2021-08-11_091343_0_main.jpg
Reading: 202

Reading: 2021-09-18_155159_0_main.jpg
Reading: 2021-09-18_155207_0_main.jpg
Reading: 2021-09-18_155216_0_main.jpg
Reading: 2021-09-18_155224_0_main.jpg
Reading: 2021-09-18_180435_0_main.jpg
Reading: 2021-09-18_191259_0_main.jpg
Reading: 2021-09-18_194447_0_main.jpg
Reading: 2021-09-19_002504_0_main.jpg
Reading: 2021-09-19_005608_0_main.jpg
Reading: 2021-09-19_013227_0_main.jpg
Reading: 2021-09-19_114943_0_main.jpg
Reading: 2021-09-19_121036_0_main.jpg
Reading: 2021-09-19_121042_0_main.jpg
Reading: 2021-09-19_121059_0_main.jpg
Reading: 2021-09-19_123809_0_main.jpg
Reading: 2021-09-19_180016_0_main.jpg
Reading: 2021-09-20_151914_0_main.jpg
Reading: 2021-09-20_152623_0_main.jpg
Reading: 2021-09-21_140253_0_main.jpg
Reading: 2021-09-23_085028_0_main.jpg
Reading: 2021-09-23_151507_0_main.jpg
Reading: 2021-09-23_224623_0_main.jpg
Reading: 2021-09-24_111124_0_main.jpg
Reading: 2021-09-25_162028_0_main.jpg
Reading: 2021-09-27_191438_0_main.jpg
Reading: 2021-09-29_123509_0_main.jpg
Reading: 202

Reading: 2021-11-16_023114_0_main.jpg
Reading: 2021-11-16_111050_0_main.jpg
Reading: 2021-11-16_132451_0_main.jpg
Reading: 2021-11-16_132535_0_main.jpg
Reading: 2021-11-16_132554_0_main.jpg
Reading: 2021-11-16_132627_0_main.jpg
Reading: 2021-11-16_154113_0_main.jpg
Reading: 2021-11-17_152830_0_main.jpg
Reading: 2021-11-17_152841_0_main.jpg
Reading: 2021-11-17_152856_0_main.jpg
Reading: 2021-11-17_152904_0_main.jpg
Reading: 2021-11-17_154412_0_main.jpg
Reading: 2021-11-17_170943_0_main.jpg
Reading: 2021-11-17_193457_0_main.jpg
Reading: 2021-11-17_194323_0_main.jpg
Reading: 2021-11-17_214528_0_main.jpg
Reading: 2021-11-17_222312_0_main.jpg
Reading: 2021-11-18_102016_0_main.jpg
Reading: 2021-11-18_105741_0_main.jpg
Reading: 2021-11-18_105757_0_main.jpg
Reading: 2021-11-18_152459_0_main.jpg
Reading: 2021-11-18_152510_0_main.jpg
Reading: 2021-11-18_153320_0_main.jpg
Reading: 2021-11-18_235520_0_main.jpg
Reading: 2021-11-19_165029_0_main.jpg
Reading: 2021-11-20_101256_0_main.jpg
Reading: 202

Reading: 2021-12-26_160435_0_main.jpg
Reading: 2021-12-27_010803_0_main.jpg
Reading: 2021-12-27_012747_0_main.jpg
Reading: 2021-12-27_205630_0_main.jpg
Reading: 2021-12-28_012115_0_main.jpg
Reading: 2021-12-28_012127_0_main.jpg
Reading: 2021-12-28_012134_0_main.jpg
Reading: 2021-12-28_012309_0_main.jpg
Reading: 2021-12-28_012314_0_main.jpg
Reading: 2021-12-28_012433_0_main.jpg
Reading: 2021-12-28_173153_0_main.jpg
Reading: 2021-12-28_194214_0_main.jpg
Reading: 2021-12-28_230322_0_main.jpg
Reading: 2021-12-28_230331_0_main.jpg
Reading: 2021-12-29_020104_0_main.jpg
Reading: 2021-12-29_205511_0_main.jpg
Reading: 2021-12-30_005237_0_main.jpg
Reading: 2021-12-30_005301_0_main.jpg
Reading: 2021-12-30_025636_0_main.jpg
Reading: 2021-12-30_120312_0_main.jpg
Reading: 2021-12-30_121358_0_main.jpg
Reading: 2021-12-30_152357_0_main.jpg
Reading: 2021-12-30_172541_0_main.jpg
Reading: 2021-12-31_224440_0_main.jpg


In [44]:
contradictary_files

{}

In [33]:
img = Image.open("C:/Users/laram/github-repos/Snapchat-Memories-Metadata/snapchat_downloads_2025_renamed/2025-08-17_184239_1.jpg")
exif_data = img._getexif()

In [35]:
exif_data.get(datetime_original_tag)

AttributeError: 'NoneType' object has no attribute 'get'

In [36]:
exif_data

In [10]:
#testing as of 15/05/26
file_source_path = "C:/Users/laram/Downloads/mydata~1778790010333/memories/2025-01-06_457FC106-E8D0-49D6-9080-BCB81E1FCDDC-main.jpg"
img = Image.open(file_source_path)
exif_data = img._getexif()
print(exif_data)


#print(has_exif_datetime(file_source_path))

{296: 2, 34665: 90, 274: 1, 282: 72.0, 283: 72.0, 40961: 1, 40962: 1242, 40963: 2208}


In [11]:
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
import os
from datetime import datetime

file_source_path = "C:/Users/laram/Downloads/mydata~1778790010333/memories/2025-01-06_457FC106-E8D0-49D6-9080-BCB81E1FCDDC-main.jpg"
path = Path(file_source_path)

img = Image.open(path)
exif = img.getexif()

# Convert tag IDs to names
exif_named = {
    TAGS.get(tag_id, tag_id): value
    for tag_id, value in exif.items()
}

date = (
    exif_named.get("DateTimeOriginal")
    or exif_named.get("DateTimeDigitized")
    or exif_named.get("DateTime")
)

if date:
    print("EXIF date:", date)
else:
    modified = datetime.fromtimestamp(os.path.getmtime(path))
    print("Filesystem modified:", modified)

Filesystem modified: 2025-01-06 05:54:53


In [15]:
exif_named.get("DateTime")

In [8]:
file_dest_path

'C:/Users/laram/Pictures/2025 Renamed Snapchats with EXIF/2025-12-29_154007_0_main.jpg'

# Note
If the image has no usable EXIF, but it has it in the filename or has file system modified date, then then use the following code:

In [ ]:
contradictary_files = {}

folder = Path(source_base)
for file in folder.rglob("*"):
    
    suffix = Path(str(file)).suffix
    if suffix.upper() not in ['.JPG', '.JPEG']:
        continue
    
    # Defining file-specific variables
    file_source_path = str(file)
    filename = Path(file_source_path).name
    
    print("Reading:", filename)
    
    try:
        img = Image.open(file_source_path)
    except:
        print("Error opening image:", file_source_path)
        continue
        
    exif_data = img._getexif()
    if not exif_data:
        print("Error opening EXIF, need to create manually:", filename)
        continue
    
    original_datetime = exif_data.get(datetime_original_tag)
    digitized_datetime = exif_data.get(datetime_digitized_tag)
    plain_datetime = exif_data.get(datetime_tag)
    filename_datetime = filename_timestamp_splitter(filename[:17])
    
    
    # Step 1: Ignore if there is no datetime found in the filename
    if not contains_real_datetime(filename[:17]):
        print("   Step 1: Failed - filename does not contain datetime data -", file_source_path)
        continue

    filename_datetime_formatted = format_exif_timestamp(filename_datetime)
    
    
    # Step 2: Ignore if there is already a datetime found in the EXIF data (but with consolidation checks)
    if has_exif_datetime(file_source_path):
        
        print("   Step 2: Failed - EXIF data already contains datetime data -", file_source_path)
        
        # Step 2.5: Flag the file if the filename datetime is different from the EXIF Datetime original
        if original_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = original_datetime
            print("   Step 2.5: Failed")

        continue
    
    # Step 3: Check that other exif datetime values don't differ from the filename datetime
    if digitized_datetime or plain_datetime:
        other_exif_datetime = digitized_datetime or plain_datetime
        if other_exif_datetime != filename_datetime_formatted:
            contradictary_files[file_source_path] = other_exif_datetime
            print("   Step 3: Failed - Contradictory dates found -", file_source_path)
            continue

            
    # Happy that the filename will be the datetime value for the exif data
    final_exif_datetime = filename_datetime_formatted
    
    # Step 4: Copy image to the destination
    file_dest_path = destination_base + filename
    try:
        shutil.copyfile(file_source_path, file_dest_path)
    except:
        print("Error copying file: " + file_path)
        print("   Step 4: Failed (Error copying file: " + file_path + ")")
        continue


    # Step 5: Add datetime values to exif data if they don't already exist
    with Image.open(file_dest_path) as new_img:
        new_exif_data = new_img._getexif()
        new_exif_dict = piexif.load(new_img.info.get("exif", b""))

        new_exif_dict["Exif"][piexif.ExifIFD.DateTimeOriginal]= final_exif_datetime

        try:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized]
        except:
            new_exif_dict["Exif"][piexif.ExifIFD.DateTimeDigitized] = final_exif_datetime

        try:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime]
        except:
            new_exif_dict["0th"][piexif.ImageIFD.DateTime] = final_exif_datetime

        exif_bytes = piexif.dump(new_exif_dict)

        new_img.save(file_dest_path, exif=exif_bytes)
        img.close()
        new_img.close()
    
    # Step 6: Validation
    if Path(file_dest_path).exists():
        
        with Image.open(file_dest_path) as verify_img:
            new_exif_data = verify_img._getexif()

        if new_exif_data.get(datetime_original_tag):
            os.remove(file_source_path)
        else:
            print("Datetime data not found in EXIF:", file_dest_path)
            continue
    else:
        print("Copy failed, not deleting:", source)
        continue
    
